# Advanced RAG Pipeline — LangChain Edition

Techniques implemented in this notebook:

| # | Technique | Purpose |
|---|-----------|---------|
| 1 | **BM25 Retrieval** | Sparse keyword matching alongside dense vectors |
| 2 | **LLM Query Rewrite** | Rephrase the user question for better retrieval |
| 3 | **HyDE** | Generate a hypothetical answer and embed it as the query |
| 4 | **Dense Vector Retrieval** | Semantic similarity via OpenAI embeddings + ChromaDB |
| 5 | **Reciprocal Rank Fusion (RRF)** | Merge ranked lists from BM25 + dense into one ranked list |
| 6 | **Cross-Encoder Reranking** | Fine-grained reranking of the fused candidates |
| 7 | **Augmented Generation** | Feed reranked context to `gpt-4o-mini` for the final answer |

Model: `gpt-4o-mini` throughout.


---
## Section 0 — Installation and Setup

In [1]:
# ---------------------------------------------------------------
# INSTALL DEPENDENCIES
# ---------------------------------------------------------------

!pip install -q \
    openai \
    langchain langchain-openai langchain-community langchain-core \
    chromadb faiss-cpu \
    rank_bm25 \
    sentence-transformers \
    tiktoken numpy requests pypdf

print("[SETUP] All dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.

In [2]:
# ---------------------------------------------------------------
# LOAD API KEY FROM COLAB SECRETS
# ---------------------------------------------------------------

import os
from google.colab import userdata

try:
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise ValueError("Secret is empty.")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("[SETUP] OPENAI_API_KEY loaded.")
except Exception as e:
    raise SystemExit(
        "\n[ERROR] Could not load OPENAI_API_KEY from Colab Secrets.\n"
        "  1. Click the key icon in the left sidebar\n"
        "  2. Add secret name: OPENAI_API_KEY\n"
        "  3. Paste your OpenAI key\n"
        "  4. Toggle 'Notebook access' ON\n"
        "  5. Re-run this cell\n"
        f"  (Error: {e})"
    )

# Optional: LangSmith tracing
# try:
#     lc_key = userdata.get("LANGCHAIN_API_KEY")
#     if lc_key:
#         os.environ["LANGCHAIN_API_KEY"] = lc_key
#         os.environ["LANGCHAIN_TRACING_V2"] = "true"
#         os.environ["LANGCHAIN_PROJECT"] = "advanced_rag"
#         print("[SETUP] LangSmith tracing enabled.")
#     else:
#         print("[SETUP] LangSmith tracing disabled.")
# except Exception:
#     print("[SETUP] LangSmith tracing disabled.")

# print("\n[SETUP] Environment configured.")


[SETUP] OPENAI_API_KEY loaded.


In [3]:
# ---------------------------------------------------------------
# INITIALIZE OPENAI CLIENT
# ---------------------------------------------------------------

from openai import OpenAI

client = OpenAI()
print("[SETUP] OpenAI client initialized.")


[SETUP] OpenAI client initialized.


---
## Section 1 — Document Loading and Chunking

Load the source document and split it into overlapping chunks using
`RecursiveCharacterTextSplitter`. Each chunk becomes one unit that can be
retrieved by BM25 or vector search.


In [4]:
# ---------------------------------------------------------------
# DEFAULT DOCUMENT
# Change RAW_DOCUMENT or upload a PDF in the next cell.
# ---------------------------------------------------------------

RAW_DOCUMENT = """
LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.

LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.
It uses a graph-based architecture where nodes represent computation steps and edges represent transitions.
LangGraph is particularly useful for building complex agent workflows that require cycles and branching logic.

LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.
It provides tracing capabilities that allow developers to see every step of an LLM chain or agent run.
LangSmith integrates seamlessly with LangChain and LangGraph.

RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved
from a knowledge base and provided to the LLM as context before generating a response.
RAG helps reduce hallucinations and allows the model to answer questions about private or recent data.

Embeddings are numerical vector representations of text. Similar texts produce similar embedding vectors.
This property allows us to find semantically similar documents using distance metrics like cosine similarity.
OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding models.

Vector stores are databases optimized for storing and searching embedding vectors.
FAISS (Facebook AI Similarity Search) is a popular open-source vector store for fast nearest-neighbor search.
Other vector stores include Chroma, Pinecone, Weaviate, and Qdrant.

Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a function that the agent can call, such as a web search, calculator, or API call.
Agents decide which tools to use and in what order based on the user's query.
"""

print("[DOCUMENT] Default document loaded.")
print(f"Characters: {len(RAW_DOCUMENT)} | Words: {len(RAW_DOCUMENT.split())}")


[DOCUMENT] Default document loaded.
Characters: 1948 | Words: 278


In [6]:
# ---------------------------------------------------------------
# OPTIONAL: Upload a PDF to replace the default document
# Skip this cell if you want to use the default text above.
# ---------------------------------------------------------------

# from google.colab import files
# from langchain_community.document_loaders import PyPDFLoader

# print("[PDF] Upload a PDF file, or skip this cell...")
# uploaded = files.upload()

# if uploaded:
#     pdf_path = list(uploaded.keys())[0]
#     loader = PyPDFLoader(pdf_path)
#     pdf_docs = loader.load()
#     RAW_DOCUMENT = "\n\n".join(
#         doc.page_content.strip() for doc in pdf_docs if doc.page_content.strip()
#     )
#     print(f"[PDF] Loaded {len(pdf_docs)} pages from: {pdf_path}")
#     print(f"[PDF] Characters: {len(RAW_DOCUMENT)} | Words: {len(RAW_DOCUMENT.split())}")
#     print("\n[PDF] Preview:\n")
#     print(RAW_DOCUMENT[:600])
# else:
#     print("[PDF] No file uploaded. Using the default document.")


In [7]:
# ---------------------------------------------------------------
# CHUNKING with RecursiveCharacterTextSplitter
# ---------------------------------------------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

raw_doc = Document(
    page_content=RAW_DOCUMENT,
    metadata={"source": "manual", "title": "LangChain Overview"}
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""]
)

lc_chunks = splitter.split_documents([raw_doc])

# Attach a stable chunk_id to every chunk for RRF bookkeeping
for i, chunk in enumerate(lc_chunks):
    chunk.metadata["chunk_id"] = i

print(f"[CHUNKING] Produced {len(lc_chunks)} chunks")
for i, chunk in enumerate(lc_chunks, 1):
    print(f"  Chunk {i:02d} ({len(chunk.page_content):3d} chars): {chunk.page_content[:80]}...")


[CHUNKING] Produced 9 chunks
  Chunk 01 (228 chars): LangChain is a framework designed to simplify the creation of applications using...
  Chunk 02 (224 chars): LangGraph is a library built on top of LangChain that enables building stateful,...
  Chunk 03 (110 chars): LangGraph is particularly useful for building complex agent workflows that requi...
  Chunk 04 (267 chars): LangSmith is a developer platform for debugging, testing, evaluating, and monito...
  Chunk 05 (293 chars): RAG stands for Retrieval-Augmented Generation. It is a technique where relevant ...
  Chunk 06 (215 chars): Embeddings are numerical vector representations of text. Similar texts produce s...
  Chunk 07 ( 94 chars): OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used emb...
  Chunk 08 (260 chars): Vector stores are databases optimized for storing and searching embedding vector...
  Chunk 09 (241 chars): Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is...

---
## Section 2 — BM25 Sparse Retriever

BM25 (Best Match 25) is a bag-of-words ranking function.  It scores documents
by term frequency and inverse document frequency without any embeddings.
It is fast, interpretable, and catches exact keyword matches that dense
retrievers sometimes miss.

`rank_bm25` is used here. Tokenization is whitespace + lowercase.


In [8]:
# ---------------------------------------------------------------
# BUILD BM25 INDEX
# ---------------------------------------------------------------

from rank_bm25 import BM25Okapi

def simple_tokenize(text: str) -> list:
    """Lower-case, whitespace tokenization. Swap in a stemmer for better recall."""
    return text.lower().split()

bm25_corpus = [simple_tokenize(chunk.page_content) for chunk in lc_chunks]
bm25_index  = BM25Okapi(bm25_corpus)

print(f"[BM25] Index built over {len(bm25_corpus)} documents.")
print(f"[BM25] Vocabulary size: {len(bm25_index.idf)}")


def bm25_retrieve(query: str, top_k: int = 10) -> list:
    """
    Return up to top_k (Document, score) tuples ranked by BM25.
    Scores are raw BM25 values (higher = more relevant).
    """
    tokens = simple_tokenize(query)
    scores = bm25_index.get_scores(tokens)
    ranked = sorted(
        zip(scores, lc_chunks),
        key=lambda x: x[0],
        reverse=True
    )
    results = [(doc, float(score)) for score, doc in ranked[:top_k]]
    return results


# Quick smoke test
print("\n[BM25] Test query: 'vector stores FAISS'")
for rank, (doc, score) in enumerate(bm25_retrieve("vector stores FAISS", top_k=3), 1):
    print(f"  Rank {rank} | BM25={score:.4f} | {doc.page_content[:90]}...")


[BM25] Index built over 9 documents.
[BM25] Vocabulary size: 185

[BM25] Test query: 'vector stores FAISS'
  Rank 1 | BM25=5.7850 | Vector stores are databases optimized for storing and searching embedding vectors.
FAISS (...
  Rank 2 | BM25=1.1469 | Embeddings are numerical vector representations of text. Similar texts produce similar emb...
  Rank 3 | BM25=0.0000 | LangChain is a framework designed to simplify the creation of applications using large lan...


---
## Section 3 — Dense Vector Retriever

OpenAI `text-embedding-3-small` embeddings are stored in ChromaDB.
Dense retrieval captures semantic similarity even when the query and document
use different vocabulary.


In [9]:
# ---------------------------------------------------------------
# EMBEDDINGS + CHROMA VECTOR STORE
# ---------------------------------------------------------------

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

print("[EMBEDDINGS] Initializing text-embedding-3-small...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"]
)

print("[VECTOR STORE] Building ChromaDB from chunks...")
chroma_store = Chroma.from_documents(
    documents=lc_chunks,
    embedding=embeddings_model,
    persist_directory="./chroma_advanced_rag"
)
print(f"[VECTOR STORE] ChromaDB ready. Indexed {len(lc_chunks)} chunks.")


def dense_retrieve(query: str, top_k: int = 10) -> list:
    """
    Return up to top_k (Document, score) tuples from ChromaDB.
    Scores are cosine similarity (higher = more relevant).
    """
    results = chroma_store.similarity_search_with_relevance_scores(query, k=top_k)
    return results   # [(Document, float), ...]


# Smoke test
print("\n[DENSE] Test query: 'What is retrieval augmented generation?'")
for rank, (doc, score) in enumerate(dense_retrieve("What is retrieval augmented generation?", top_k=3), 1):
    print(f"  Rank {rank} | Cosine={score:.4f} | {doc.page_content[:90]}...")


[EMBEDDINGS] Initializing text-embedding-3-small...
[VECTOR STORE] Building ChromaDB from chunks...
[VECTOR STORE] ChromaDB ready. Indexed 9 chunks.

[DENSE] Test query: 'What is retrieval augmented generation?'
  Rank 1 | Cosine=0.4421 | RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents ...
  Rank 2 | Cosine=0.0192 | OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding mod...
  Rank 3 | Cosine=-0.0579 | Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a functio...


/tmp/ipykernel_2350/594244655.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'chunk_id': 4, 'title': 'LangChain Overview', 'source': 'manual'}, page_content='RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved\nfrom a knowledge base and provided to the LLM as context before generating a response.\nRAG helps reduce hallucinations and allows the model to answer questions about private or recent data.'), 0.4420657092373338), (Document(metadata={'source': 'manual', 'chunk_id': 6, 'title': 'LangChain Overview'}, page_content="OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding models."), 0.01917599024975769), (Document(metadata={'title': 'LangChain Overview', 'source': 'manual', 'chunk_id': 8}, page_content="Agents are LLM-powered systems that can use tools to accomplish tasks.\nA tool is a function that the agent can call, such as a web search, calculator, or API c

---
## Section 4 — LLM Query Rewrite

Before retrieval, the original user question is rewritten by `gpt-4o-mini`
into a cleaner, more self-contained retrieval query.

This helps when the user's question is:
- Conversational or ambiguous
- Using pronouns that lack referents
- Missing context that retrieval needs

The rewritten query is used for both BM25 and dense retrieval.


In [10]:
# ---------------------------------------------------------------
# LLM QUERY REWRITE
# ---------------------------------------------------------------

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rewrite_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"]
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """You are an expert at improving search queries for document retrieval.

Rewrite the user question below into a clear, specific, standalone search query.
- Remove conversational filler
- Expand abbreviations if obvious
- Make the query self-contained
- Return ONLY the rewritten query, no explanation

Original question: {question}

Rewritten query:"""
)

rewrite_chain = rewrite_prompt | rewrite_llm | StrOutputParser()


def rewrite_query(question: str, verbose: bool = True) -> str:
    """Rewrite the user question into a better retrieval query."""
    rewritten = rewrite_chain.invoke({"question": question}).strip()
    if verbose:
        print(f"  [REWRITE] Original : {question}")
        print(f"  [REWRITE] Rewritten: {rewritten}")
    return rewritten


# Smoke test
rewrite_query("What does that LangSmith thing do exactly?")


  [REWRITE] Original : What does that LangSmith thing do exactly?
  [REWRITE] Rewritten: What are the features and functions of LangSmith?


'What are the features and functions of LangSmith?'

---
## Section 5 — HyDE (Hypothetical Document Embeddings)

**HyDE** (Gao et al., 2022) generates a hypothetical answer to the query
using the LLM, then embeds that hypothetical answer instead of (or in
addition to) the original query.

The intuition: a generated answer lives in the same embedding space as real
document passages, so it bridges the vocabulary gap between short queries
and longer passages.

Here HyDE is used as an additional dense retrieval path. Its results are
fused with BM25 and standard dense results via RRF.


In [11]:
# ---------------------------------------------------------------
# HyDE — HYPOTHETICAL DOCUMENT GENERATION
# ---------------------------------------------------------------

hyde_prompt = ChatPromptTemplate.from_template(
    """You are an expert assistant. Write a short, factual passage (2-4 sentences)
that would directly answer the following question.
Do NOT reference the question itself. Write as if you are the document being retrieved.

Question: {question}

Hypothetical passage:"""
)

hyde_chain = hyde_prompt | rewrite_llm | StrOutputParser()


def hyde_retrieve(question: str, top_k: int = 10, verbose: bool = True) -> list:
    """
    Generate a hypothetical answer, embed it, and retrieve against ChromaDB.
    Returns list of (Document, score) tuples.
    """
    hypothetical_doc = hyde_chain.invoke({"question": question}).strip()
    if verbose:
        print(f"  [HyDE] Hypothetical passage: {hypothetical_doc[:150]}...")

    results = chroma_store.similarity_search_with_relevance_scores(
        hypothetical_doc,
        k=top_k
    )
    return results


# Smoke test
print("[HyDE] Test query: 'How do agents decide which tool to use?'")
for rank, (doc, score) in enumerate(hyde_retrieve("How do agents decide which tool to use?", top_k=3), 1):
    print(f"  Rank {rank} | Cosine={score:.4f} | {doc.page_content[:90]}...")


[HyDE] Test query: 'How do agents decide which tool to use?'
  [HyDE] Hypothetical passage: Agents typically evaluate the specific needs of a task, the capabilities of available tools, and the context in which they are operating. They conside...
  Rank 1 | Cosine=0.5334 | Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a functio...
  Rank 2 | Cosine=0.1057 | LangGraph is particularly useful for building complex agent workflows that require cycles ...
  Rank 3 | Cosine=0.0440 | RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents ...


---
## Section 6 — Reciprocal Rank Fusion (RRF)

RRF (Cormack et al., 2009) merges multiple ranked lists into a single
combined ranking without needing to normalize scores across retrievers.

Formula for a document `d` across `k` ranked lists:

```
RRF(d) = sum over all lists of  1 / (k_constant + rank_in_list)
```

`k_constant = 60` is the standard value. Documents appearing high in
multiple lists get the largest fused scores.

We fuse three lists: BM25, dense (with rewritten query), and HyDE dense.


In [12]:
# ---------------------------------------------------------------
# RECIPROCAL RANK FUSION
# ---------------------------------------------------------------

def reciprocal_rank_fusion(
    ranked_lists: list,
    k_constant: int = 60,
    top_k: int = 15
) -> list:
    """
    Fuse multiple ranked lists of (Document, score) tuples.

    Args:
        ranked_lists : list of lists, each inner list is [(Document, score), ...]
                       ordered best-first.
        k_constant   : RRF constant (default 60).
        top_k        : number of results to return.

    Returns:
        List of (Document, rrf_score) sorted descending, deduplicated by chunk_id.
    """
    rrf_scores: dict = {}   # chunk_id -> rrf_score
    id_to_doc:  dict = {}   # chunk_id -> Document

    for ranked_list in ranked_lists:
        for rank, (doc, _score) in enumerate(ranked_list, start=1):
            cid = doc.metadata.get("chunk_id", doc.page_content[:60])
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k_constant + rank)
            id_to_doc[cid] = doc

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [(id_to_doc[cid], score) for cid, score in fused[:top_k]]


def multi_retrieve(question: str, top_k_each: int = 10, verbose: bool = True) -> list:
    """
    Run BM25, dense (rewritten query), and HyDE dense, then fuse with RRF.
    Returns list of (Document, rrf_score).
    """
    print(f"\n[RETRIEVE] Original question: {question}")

    # Step 1 – rewrite
    rewritten = rewrite_query(question, verbose=verbose)

    # Step 2 – BM25 on rewritten query
    bm25_results  = bm25_retrieve(rewritten, top_k=top_k_each)
    if verbose:
        print(f"  [BM25] Retrieved {len(bm25_results)} candidates")

    # Step 3 – dense on rewritten query
    dense_results = dense_retrieve(rewritten, top_k=top_k_each)
    if verbose:
        print(f"  [DENSE] Retrieved {len(dense_results)} candidates")

    # Step 4 – HyDE dense on original question
    hyde_results  = hyde_retrieve(question, top_k=top_k_each, verbose=verbose)
    if verbose:
        print(f"  [HyDE] Retrieved {len(hyde_results)} candidates")

    # Step 5 – RRF fusion
    fused = reciprocal_rank_fusion(
        [bm25_results, dense_results, hyde_results],
        k_constant=60,
        top_k=top_k_each
    )
    if verbose:
        print(f"  [RRF] Fused into {len(fused)} unique candidates")

    return fused


# Smoke test
print("=" * 60)
print("RRF FUSION SMOKE TEST")
print("=" * 60)
fused_results = multi_retrieve("What is LangSmith used for?")
print("\n[RRF] Top-5 after fusion:")
for rank, (doc, score) in enumerate(fused_results[:5], 1):
    print(f"  Rank {rank} | RRF={score:.5f} | {doc.page_content[:90]}...")


RRF FUSION SMOKE TEST

[RETRIEVE] Original question: What is LangSmith used for?
  [REWRITE] Original : What is LangSmith used for?
  [REWRITE] Rewritten: LangSmith application and use cases
  [BM25] Retrieved 9 candidates


/tmp/ipykernel_2350/594244655.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'source': 'manual', 'title': 'LangChain Overview', 'chunk_id': 3}, page_content='LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.\nIt provides tracing capabilities that allow developers to see every step of an LLM chain or agent run.\nLangSmith integrates seamlessly with LangChain and LangGraph.'), 0.5240207197821009), (Document(metadata={'source': 'manual', 'title': 'LangChain Overview', 'chunk_id': 0}, page_content='LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).\nIt provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.'), 0.14244498543199502), (Document(metadata={'chunk_id': 8, 'source': 'manual', 'title': 'LangChain Overview'}, page_content="Agents are LLM-powered systems that can use tools t

  [DENSE] Retrieved 9 candidates
  [HyDE] Hypothetical passage: LangSmith is a platform designed for developers and teams to build, manage, and deploy language models efficiently. It provides tools for training, fi...
  [HyDE] Retrieved 9 candidates
  [RRF] Fused into 9 unique candidates

[RRF] Top-5 after fusion:
  Rank 1 | RRF=0.04918 | LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM a...
  Rank 2 | RRF=0.04763 | Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a functio...
  Rank 3 | RRF=0.04696 | LangChain is a framework designed to simplify the creation of applications using large lan...
  Rank 4 | RRF=0.04642 | LangGraph is a library built on top of LangChain that enables building stateful, multi-act...
  Rank 5 | RRF=0.04618 | OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding mod...


/tmp/ipykernel_2350/3098778183.py:27: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'source': 'manual', 'title': 'LangChain Overview', 'chunk_id': 3}, page_content='LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.\nIt provides tracing capabilities that allow developers to see every step of an LLM chain or agent run.\nLangSmith integrates seamlessly with LangChain and LangGraph.'), 0.6498917770977037), (Document(metadata={'title': 'LangChain Overview', 'source': 'manual', 'chunk_id': 0}, page_content='LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).\nIt provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.'), 0.2890406596847688), (Document(metadata={'source': 'manual', 'chunk_id': 1, 'title': 'LangChain Overview'}, page_content='LangGraph is a library built on top of LangChain th

---
## Section 7 — Cross-Encoder Reranking

A **cross-encoder** takes the query and a candidate document as a *pair* and
outputs a single relevance score. Unlike bi-encoders (used for embedding
similarity), cross-encoders can model fine-grained interaction between query
and document tokens.

They are too slow to run over all documents in a corpus, so they are applied
only to the small candidate set produced by RRF — this is the classic
**retrieve-then-rerank** pattern.

Model: `cross-encoder/ms-marco-MiniLM-L-6-v2` from `sentence-transformers`.


In [13]:
# ---------------------------------------------------------------
# LOAD CROSS-ENCODER MODEL (downloaded once, cached by HuggingFace)
# ---------------------------------------------------------------

from sentence_transformers import CrossEncoder

print("[CROSS-ENCODER] Loading cross-encoder/ms-marco-MiniLM-L-6-v2 ...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("[CROSS-ENCODER] Model ready.")


[CROSS-ENCODER] Loading cross-encoder/ms-marco-MiniLM-L-6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[CROSS-ENCODER] Model ready.


In [14]:
# ---------------------------------------------------------------
# CROSS-ENCODER RERANKING
# ---------------------------------------------------------------

def cross_encoder_rerank(
    query: str,
    candidates: list,
    top_k: int = 5
) -> list:
    """
    Rerank a list of (Document, _score) tuples using the cross-encoder.

    Args:
        query      : the original user question.
        candidates : list of (Document, any_score) — order does not matter here.
        top_k      : how many to return after reranking.

    Returns:
        List of (Document, cross_encoder_score) sorted descending.
    """
    if not candidates:
        return []

    pairs = [(query, doc.page_content) for doc, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)

    reranked = sorted(
        zip(ce_scores, [doc for doc, _ in candidates]),
        key=lambda x: x[0],
        reverse=True
    )
    return [(doc, float(score)) for score, doc in reranked[:top_k]]


# Smoke test
print("[CE RERANK] Smoke test: 'What is RAG?'")
fused_candidates = multi_retrieve("What is RAG?", top_k_each=10, verbose=False)
reranked = cross_encoder_rerank("What is RAG?", fused_candidates, top_k=5)
for rank, (doc, score) in enumerate(reranked, 1):
    print(f"  Rank {rank} | CE Score={score:.4f} | {doc.page_content[:90]}...")


[CE RERANK] Smoke test: 'What is RAG?'

[RETRIEVE] Original question: What is RAG?


/tmp/ipykernel_2350/594244655.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'chunk_id': 4, 'title': 'LangChain Overview', 'source': 'manual'}, page_content='RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved\nfrom a knowledge base and provided to the LLM as context before generating a response.\nRAG helps reduce hallucinations and allows the model to answer questions about private or recent data.'), 0.4669669102745402), (Document(metadata={'title': 'LangChain Overview', 'source': 'manual', 'chunk_id': 6}, page_content="OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding models."), 0.02699659087204509), (Document(metadata={'title': 'LangChain Overview', 'chunk_id': 8, 'source': 'manual'}, page_content="Agents are LLM-powered systems that can use tools to accomplish tasks.\nA tool is a function that the agent can call, such as a web search, calculator, or API c

  Rank 1 | CE Score=9.0629 | RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents ...
  Rank 2 | CE Score=-9.3976 | LangChain is a framework designed to simplify the creation of applications using large lan...
  Rank 3 | CE Score=-9.6286 | LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM a...
  Rank 4 | CE Score=-10.1348 | LangGraph is a library built on top of LangChain that enables building stateful, multi-act...
  Rank 5 | CE Score=-10.4075 | Vector stores are databases optimized for storing and searching embedding vectors.
FAISS (...


---
## Section 8 — Semantic Cache with TTL

### What this cache does

| Property | Detail |
|----------|--------|
| **Cache type** | In-memory semantic cache (per session) |
| **Key** | Normalized (lowercased, stripped) question string |
| **Lookup** | Exact string match first; then cosine similarity over stored embeddings |
| **TTL** | Configurable per-entry time-to-live (default 300 s). Expired entries are evicted on every access. |
| **Similarity threshold** | Configurable (default 0.92). On a semantic hit the log shows the match percentage and the original matched question. |
| **Miss path** | Full pipeline runs; result is stored with a fresh timestamp. |

Every call prints exactly one log line:

```
[CACHE] EXACT HIT    | age=12.3s / TTL=300s | cache_type=exact_string    | question: "..."
[CACHE] SEMANTIC HIT | age=45.1s / TTL=300s | similarity=96.4% | cache_type=semantic_cosine | matched: "..."
[CACHE] MISS         | cache_type=N/A        | no match above threshold=0.92
[CACHE] STORED       | TTL=300s              | key: "..."
[CACHE] EXPIRED      | evicted key: "..."
```


In [15]:
# ---------------------------------------------------------------
# SEMANTIC CACHE WITH TTL
# ---------------------------------------------------------------

import time
import numpy as np
from dataclasses import dataclass, field
from typing import Optional

# ---------------------------------------------------------------
# CONFIG  (change these to tune cache behaviour)
# ---------------------------------------------------------------
CACHE_TTL_SECONDS          = 300    # seconds before an entry expires
CACHE_SIMILARITY_THRESHOLD = 0.92   # cosine similarity required for semantic hit
# ---------------------------------------------------------------


@dataclass
class CacheEntry:
    question:   str
    answer:     str
    embedding:  np.ndarray
    timestamp:  float = field(default_factory=time.time)

    def age(self) -> float:
        return time.time() - self.timestamp

    def is_expired(self) -> bool:
        return self.age() > CACHE_TTL_SECONDS


class SemanticCache:
    """
    In-memory semantic cache for RAG answers.

    Lookup order
    ------------
    1. Exact string match  (cache_type = exact_string)
       No embedding call needed; O(1).

    2. Cosine similarity   (cache_type = semantic_cosine)
       Embeds the query, compares against all stored embeddings.
       Returns the best match if it exceeds CACHE_SIMILARITY_THRESHOLD.

    3. Miss
       The full pipeline runs; the result is stored for future lookups.
    """

    def __init__(self):
        self._store: dict = {}        # norm_question -> CacheEntry
        self._hits_exact    = 0
        self._hits_semantic = 0
        self._misses        = 0
        self._evictions     = 0

    # -- internal helpers --------------------------------------------------

    @staticmethod
    def _normalize(text: str) -> str:
        return text.strip().lower()

    @staticmethod
    def _cosine(a: np.ndarray, b: np.ndarray) -> float:
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        return float(np.dot(a, b) / denom) if denom else 0.0

    def _evict_expired(self):
        expired = [k for k, v in self._store.items() if v.is_expired()]
        for k in expired:
            print(f"  [CACHE] EXPIRED      | evicted key: \"{self._store[k].question}\"")
            del self._store[k]
            self._evictions += 1

    # -- public API --------------------------------------------------------

    def get(self, question: str) -> Optional[str]:
        """
        Return cached answer, or None on miss.
        Always prints a one-line cache log.
        """
        self._evict_expired()
        norm = self._normalize(question)

        # 1. Exact match
        if norm in self._store:
            entry = self._store[norm]
            self._hits_exact += 1
            print(
                f"  [CACHE] EXACT HIT    | "
                f"age={entry.age():.1f}s / TTL={CACHE_TTL_SECONDS}s | "
                f"cache_type=exact_string | "
                f"question: \"{entry.question}\""
            )
            return entry.answer

        # 2. Semantic match
        if self._store:
            query_emb  = np.array(embeddings_model.embed_query(question))
            best_key, best_score = None, -1.0
            for key, entry in self._store.items():
                score = self._cosine(query_emb, entry.embedding)
                if score > best_score:
                    best_score, best_key = score, key

            if best_score >= CACHE_SIMILARITY_THRESHOLD:
                entry = self._store[best_key]
                self._hits_semantic += 1
                print(
                    f"  [CACHE] SEMANTIC HIT | "
                    f"age={entry.age():.1f}s / TTL={CACHE_TTL_SECONDS}s | "
                    f"similarity={best_score*100:.1f}% | "
                    f"cache_type=semantic_cosine | "
                    f"matched: \"{entry.question}\""
                )
                return entry.answer

        # 3. Miss
        self._misses += 1
        print(
            f"  [CACHE] MISS         | "
            f"cache_type=N/A | "
            f"no match above threshold={CACHE_SIMILARITY_THRESHOLD}"
        )
        return None

    def put(self, question: str, answer: str):
        """Store a question-answer pair with the current timestamp."""
        norm = self._normalize(question)
        emb  = np.array(embeddings_model.embed_query(question))
        self._store[norm] = CacheEntry(question=question, answer=answer, embedding=emb)
        print(f"  [CACHE] STORED       | TTL={CACHE_TTL_SECONDS}s | key: \"{question}\"")

    def stats(self):
        """Print a session summary table."""
        total    = self._hits_exact + self._hits_semantic + self._misses
        hit_rate = (self._hits_exact + self._hits_semantic) / total * 100 if total else 0.0
        print(f"\n{'='*58}")
        print(f"  CACHE STATISTICS")
        print(f"{'='*58}")
        print(f"  Exact hits      : {self._hits_exact}")
        print(f"  Semantic hits   : {self._hits_semantic}")
        print(f"  Misses          : {self._misses}")
        print(f"  Evictions (TTL) : {self._evictions}")
        print(f"  Total lookups   : {total}")
        print(f"  Hit rate        : {hit_rate:.1f}%")
        print(f"  Live entries    : {len(self._store)}")
        print(f"  TTL setting     : {CACHE_TTL_SECONDS}s")
        print(f"  Sim threshold   : {CACHE_SIMILARITY_THRESHOLD}")
        print(f"{'='*58}\n")

    def clear(self):
        self._store.clear()
        print("[CACHE] All entries cleared.")


rag_cache = SemanticCache()
print("[CACHE] SemanticCache initialized.")
print(f"[CACHE] TTL={CACHE_TTL_SECONDS}s  |  similarity_threshold={CACHE_SIMILARITY_THRESHOLD}")


[CACHE] SemanticCache initialized.
[CACHE] TTL=300s  |  similarity_threshold=0.92


---
## Section 9 — Full Advanced RAG Pipeline

The complete end-to-end flow:

```
User Question
    |
    v
LLM Query Rewrite  --------->  BM25 Retrieval
                   --------->  Dense Retrieval (rewritten query)
                   --------->  HyDE Dense Retrieval (original question)
                                    |
                                    v
                           Reciprocal Rank Fusion
                                    |
                                    v
                        Cross-Encoder Reranking (top-5)
                                    |
                                    v
                    Augmented Generation (gpt-4o-mini)
                                    |
                                    v
                               Final Answer
```


In [16]:
# ---------------------------------------------------------------
# LLM FOR GENERATION
# ---------------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

generation_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"]
)

generation_prompt = ChatPromptTemplate.from_template(
    """You are a precise and helpful assistant.

Use ONLY the context passages below to answer the question.
If the context does not contain enough information to answer, say:
"I cannot answer based on the provided context."

Do not fabricate information.

Context passages (ordered by relevance, most relevant first):
{context}

Question: {question}

Answer:"""
)

generation_chain = generation_prompt | generation_llm | StrOutputParser()

print("[LLM] Generation chain ready (gpt-4o-mini).")


[LLM] Generation chain ready (gpt-4o-mini).


In [17]:
# ---------------------------------------------------------------
# FULL ADVANCED RAG FUNCTION  (with semantic cache)
# ---------------------------------------------------------------

def advanced_rag(
    question: str,
    top_k_retrieval: int = 10,
    top_k_rerank: int = 5,
    verbose: bool = True
) -> str:
    """
    Run the full advanced RAG pipeline with semantic cache.

    Pipeline
    --------
    Step 0  Cache lookup   — exact match, then semantic similarity
    Step 1  Query Rewrite  — gpt-4o-mini rewrites the question
    Step 2  BM25           — sparse keyword retrieval
    Step 3  Dense          — cosine similarity via ChromaDB
    Step 4  HyDE           — hypothetical document dense retrieval
    Step 5  RRF            — fuse all three ranked lists
    Step 6  Cross-Encoder  — rerank top candidates
    Step 7  Generation     — gpt-4o-mini produces the answer
    Step 8  Cache store    — save result for future hits

    Args
    ----
    question        : user's natural-language question.
    top_k_retrieval : candidates per retriever before RRF.
    top_k_rerank    : documents passed to the LLM after reranking.
    verbose         : print step-by-step logs.

    Returns
    -------
    The generated answer as a string.
    """
    print(f"\n{'='*65}")
    print(f"ADVANCED RAG PIPELINE")
    print(f"{'='*65}")
    print(f"Question: {question}")

    # ------------------------------------------------------------------
    # STEP 0: Cache lookup
    # ------------------------------------------------------------------
    print("\n--- Step 0: Cache Lookup ---")
    cached_answer = rag_cache.get(question)
    if cached_answer is not None:
        print(f"\n[ANSWER] (served from cache)\n{cached_answer}")
        print(f"{'='*65}\n")
        return cached_answer

    # ------------------------------------------------------------------
    # STEPS 1-3: Multi-retrieval + RRF
    # ------------------------------------------------------------------
    print("\n--- Steps 1-3: Query Rewrite + Multi-Retrieval + RRF ---")
    fused = multi_retrieve(question, top_k_each=top_k_retrieval, verbose=verbose)

    # ------------------------------------------------------------------
    # STEP 4: Cross-Encoder reranking
    # ------------------------------------------------------------------
    print(f"\n--- Step 4: Cross-Encoder Reranking (top {top_k_rerank}) ---")
    reranked = cross_encoder_rerank(question, fused, top_k=top_k_rerank)

    if verbose:
        print("  Final context passages:")
        for rank, (doc, score) in enumerate(reranked, 1):
            print(f"    Rank {rank} | CE={score:.4f} | {doc.page_content[:80]}...")

    # ------------------------------------------------------------------
    # STEP 5: Augmented Generation
    # ------------------------------------------------------------------
    print("\n--- Step 5: Augmented Generation (gpt-4o-mini) ---")
    context_text = "\n\n---\n\n".join(
        f"[Passage {i}] {doc.page_content}"
        for i, (doc, _) in enumerate(reranked, 1)
    )

    answer = generation_chain.invoke({
        "context": context_text,
        "question": question
    })

    # ------------------------------------------------------------------
    # STEP 6: Store result in cache
    # ------------------------------------------------------------------
    print("\n--- Step 6: Cache Store ---")
    rag_cache.put(question, answer)

    print(f"\n[ANSWER]")
    print(answer)
    print(f"{'='*65}\n")
    return answer


---
## Section 10 — Example Queries

Run the full pipeline on several questions to see each stage in action.


### Cache Demo

The first call is a cache miss and runs the full pipeline.
The second call is an exact hit.
The third call is a paraphrase and should be a semantic hit.


In [18]:
# ---------------------------------------------------------------
# CACHE DEMO — miss, exact hit, semantic hit
# ---------------------------------------------------------------

Q1 = "What is RAG and why is it useful?"
Q2 = "What is RAG and why is it useful?"       # identical -> exact hit
Q3 = "Can you explain RAG and its benefits?"   # paraphrase -> semantic hit

print("\n--- CALL 1: First time (expect MISS) ---")
_ = advanced_rag(Q1, verbose=False)

print("\n--- CALL 2: Identical question (expect EXACT HIT) ---")
_ = advanced_rag(Q2, verbose=False)

print("\n--- CALL 3: Paraphrased question (expect SEMANTIC HIT) ---")
_ = advanced_rag(Q3, verbose=False)

print("\n--- CACHE STATS after demo ---")
rag_cache.stats()



--- CALL 1: First time (expect MISS) ---

ADVANCED RAG PIPELINE
Question: What is RAG and why is it useful?

--- Step 0: Cache Lookup ---
  [CACHE] MISS         | cache_type=N/A | no match above threshold=0.92

--- Steps 1-3: Query Rewrite + Multi-Retrieval + RRF ---

[RETRIEVE] Original question: What is RAG and why is it useful?


/tmp/ipykernel_2350/594244655.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'chunk_id': 4, 'title': 'LangChain Overview', 'source': 'manual'}, page_content='RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved\nfrom a knowledge base and provided to the LLM as context before generating a response.\nRAG helps reduce hallucinations and allows the model to answer questions about private or recent data.'), 0.4661372495616033), (Document(metadata={'title': 'LangChain Overview', 'source': 'manual', 'chunk_id': 6}, page_content="OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding models."), 0.027756161375908417), (Document(metadata={'source': 'manual', 'chunk_id': 7, 'title': 'LangChain Overview'}, page_content='Vector stores are databases optimized for storing and searching embedding vectors.\nFAISS (Facebook AI Similarity Search) is a popular open-source vector store


--- Step 4: Cross-Encoder Reranking (top 5) ---

--- Step 5: Augmented Generation (gpt-4o-mini) ---

--- Step 6: Cache Store ---
  [CACHE] STORED       | TTL=300s | key: "What is RAG and why is it useful?"

[ANSWER]
RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved from a knowledge base and provided to the LLM as context before generating a response. RAG helps reduce hallucinations and allows the model to answer questions about private or recent data.


--- CALL 2: Identical question (expect EXACT HIT) ---

ADVANCED RAG PIPELINE
Question: What is RAG and why is it useful?

--- Step 0: Cache Lookup ---
  [CACHE] EXACT HIT    | age=0.0s / TTL=300s | cache_type=exact_string | question: "What is RAG and why is it useful?"

[ANSWER] (served from cache)
RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved from a knowledge base and provided to the LLM as context before generating a respon

/tmp/ipykernel_2350/594244655.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'title': 'LangChain Overview', 'chunk_id': 4, 'source': 'manual'}, page_content='RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved\nfrom a knowledge base and provided to the LLM as context before generating a response.\nRAG helps reduce hallucinations and allows the model to answer questions about private or recent data.'), 0.42437284155409705), (Document(metadata={'chunk_id': 6, 'title': 'LangChain Overview', 'source': 'manual'}, page_content="OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding models."), -0.03139019701336965), (Document(metadata={'chunk_id': 8, 'title': 'LangChain Overview', 'source': 'manual'}, page_content="Agents are LLM-powered systems that can use tools to accomplish tasks.\nA tool is a function that the agent can call, such as a web search, calculator, or API


--- Step 4: Cross-Encoder Reranking (top 5) ---

--- Step 5: Augmented Generation (gpt-4o-mini) ---

--- Step 6: Cache Store ---
  [CACHE] STORED       | TTL=300s | key: "Can you explain RAG and its benefits?"

[ANSWER]
RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved from a knowledge base and provided to the LLM as context before generating a response. The benefits of RAG include reducing hallucinations and enabling the model to answer questions about private or recent data.


--- CACHE STATS after demo ---

  CACHE STATISTICS
  Exact hits      : 1
  Semantic hits   : 0
  Misses          : 2
  Evictions (TTL) : 0
  Total lookups   : 3
  Hit rate        : 33.3%
  Live entries    : 2
  TTL setting     : 300s
  Sim threshold   : 0.92



In [19]:
# ---------------------------------------------------------------
# EXAMPLE 1
# ---------------------------------------------------------------
answer1 = advanced_rag(
    "What is RAG and why is it useful?",
    top_k_retrieval=10,
    top_k_rerank=5,
    verbose=True
)



ADVANCED RAG PIPELINE
Question: What is RAG and why is it useful?

--- Step 0: Cache Lookup ---
  [CACHE] EXACT HIT    | age=7.6s / TTL=300s | cache_type=exact_string | question: "What is RAG and why is it useful?"

[ANSWER] (served from cache)
RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved from a knowledge base and provided to the LLM as context before generating a response. RAG helps reduce hallucinations and allows the model to answer questions about private or recent data.



In [20]:
# ---------------------------------------------------------------
# EXAMPLE 2
# ---------------------------------------------------------------
answer2 = advanced_rag(
    "How does LangGraph differ from LangChain?",
    top_k_retrieval=10,
    top_k_rerank=5,
    verbose=True
)



ADVANCED RAG PIPELINE
Question: How does LangGraph differ from LangChain?

--- Step 0: Cache Lookup ---
  [CACHE] MISS         | cache_type=N/A | no match above threshold=0.92

--- Steps 1-3: Query Rewrite + Multi-Retrieval + RRF ---

[RETRIEVE] Original question: How does LangGraph differ from LangChain?
  [REWRITE] Original : How does LangGraph differ from LangChain?
  [REWRITE] Rewritten: Differences between LangGraph and LangChain
  [BM25] Retrieved 9 candidates
  [DENSE] Retrieved 9 candidates


/tmp/ipykernel_2350/594244655.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'source': 'manual', 'chunk_id': 1, 'title': 'LangChain Overview'}, page_content='LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.\nIt uses a graph-based architecture where nodes represent computation steps and edges represent transitions.'), 0.6021556242715376), (Document(metadata={'title': 'LangChain Overview', 'chunk_id': 0, 'source': 'manual'}, page_content='LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).\nIt provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.'), 0.4118265676893742), (Document(metadata={'title': 'LangChain Overview', 'source': 'manual', 'chunk_id': 3}, page_content='LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applica

  [HyDE] Hypothetical passage: LangGraph is designed to facilitate the creation and management of complex workflows by integrating various language models and data sources, allowing...


/tmp/ipykernel_2350/3098778183.py:27: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'chunk_id': 1, 'source': 'manual', 'title': 'LangChain Overview'}, page_content='LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.\nIt uses a graph-based architecture where nodes represent computation steps and edges represent transitions.'), 0.6925048961936738), (Document(metadata={'title': 'LangChain Overview', 'chunk_id': 2, 'source': 'manual'}, page_content='LangGraph is particularly useful for building complex agent workflows that require cycles and branching logic.'), 0.5395016787086373), (Document(metadata={'chunk_id': 0, 'source': 'manual', 'title': 'LangChain Overview'}, page_content='LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).\nIt provides tools for chaining prompts, managing memory, integrating with external data sources, and b

  [HyDE] Retrieved 9 candidates
  [RRF] Fused into 9 unique candidates

--- Step 4: Cross-Encoder Reranking (top 5) ---
  Final context passages:
    Rank 1 | CE=5.8206 | LangGraph is a library built on top of LangChain that enables building stateful,...
    Rank 2 | CE=3.4154 | LangChain is a framework designed to simplify the creation of applications using...
    Rank 3 | CE=2.1826 | LangGraph is particularly useful for building complex agent workflows that requi...
    Rank 4 | CE=1.7427 | LangSmith is a developer platform for debugging, testing, evaluating, and monito...
    Rank 5 | CE=-11.0643 | RAG stands for Retrieval-Augmented Generation. It is a technique where relevant ...

--- Step 5: Augmented Generation (gpt-4o-mini) ---

--- Step 6: Cache Store ---
  [CACHE] STORED       | TTL=300s | key: "How does LangGraph differ from LangChain?"

[ANSWER]
LangGraph is built on top of LangChain and specifically enables building stateful, multi-actor applications with LLMs using a graph

In [21]:
# ---------------------------------------------------------------
# EXAMPLE 3
# ---------------------------------------------------------------
answer3 = advanced_rag(
    "What does LangSmith let developers do?",
    top_k_retrieval=10,
    top_k_rerank=5,
    verbose=True
)



ADVANCED RAG PIPELINE
Question: What does LangSmith let developers do?

--- Step 0: Cache Lookup ---
  [CACHE] MISS         | cache_type=N/A | no match above threshold=0.92

--- Steps 1-3: Query Rewrite + Multi-Retrieval + RRF ---

[RETRIEVE] Original question: What does LangSmith let developers do?
  [REWRITE] Original : What does LangSmith let developers do?
  [REWRITE] Rewritten: LangSmith features and capabilities for developers
  [BM25] Retrieved 9 candidates


/tmp/ipykernel_2350/594244655.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'title': 'LangChain Overview', 'source': 'manual', 'chunk_id': 3}, page_content='LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.\nIt provides tracing capabilities that allow developers to see every step of an LLM chain or agent run.\nLangSmith integrates seamlessly with LangChain and LangGraph.'), 0.4851460682958957), (Document(metadata={'chunk_id': 0, 'source': 'manual', 'title': 'LangChain Overview'}, page_content='LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).\nIt provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.'), 0.10573769348361395), (Document(metadata={'chunk_id': 8, 'title': 'LangChain Overview', 'source': 'manual'}, page_content="Agents are LLM-powered systems that can use tools t

  [DENSE] Retrieved 9 candidates
  [HyDE] Hypothetical passage: LangSmith allows developers to create, manage, and deploy machine learning models with ease. It provides tools for data preparation, model training, a...


/tmp/ipykernel_2350/3098778183.py:27: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'chunk_id': 3, 'source': 'manual', 'title': 'LangChain Overview'}, page_content='LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.\nIt provides tracing capabilities that allow developers to see every step of an LLM chain or agent run.\nLangSmith integrates seamlessly with LangChain and LangGraph.'), 0.6397726559450262), (Document(metadata={'chunk_id': 0, 'title': 'LangChain Overview', 'source': 'manual'}, page_content='LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).\nIt provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.'), 0.2359906743451975), (Document(metadata={'title': 'LangChain Overview', 'chunk_id': 1, 'source': 'manual'}, page_content='LangGraph is a library built on top of LangChain th

  [HyDE] Retrieved 9 candidates
  [RRF] Fused into 9 unique candidates

--- Step 4: Cross-Encoder Reranking (top 5) ---
  Final context passages:
    Rank 1 | CE=8.0142 | LangSmith is a developer platform for debugging, testing, evaluating, and monito...
    Rank 2 | CE=-0.6385 | LangChain is a framework designed to simplify the creation of applications using...
    Rank 3 | CE=-3.0658 | LangGraph is a library built on top of LangChain that enables building stateful,...
    Rank 4 | CE=-3.7034 | LangGraph is particularly useful for building complex agent workflows that requi...
    Rank 5 | CE=-10.8692 | RAG stands for Retrieval-Augmented Generation. It is a technique where relevant ...

--- Step 5: Augmented Generation (gpt-4o-mini) ---

--- Step 6: Cache Store ---
  [CACHE] STORED       | TTL=300s | key: "What does LangSmith let developers do?"

[ANSWER]
LangSmith lets developers debug, test, evaluate, and monitor LLM applications, and it provides tracing capabilities to see every st

In [22]:
# ---------------------------------------------------------------
# EXAMPLE 4 — Out-of-corpus question (should say I don't know)
# ---------------------------------------------------------------
answer4 = advanced_rag(
    "What is the capital of France?",
    top_k_retrieval=10,
    top_k_rerank=5,
    verbose=True
)



ADVANCED RAG PIPELINE
Question: What is the capital of France?

--- Step 0: Cache Lookup ---
  [CACHE] MISS         | cache_type=N/A | no match above threshold=0.92

--- Steps 1-3: Query Rewrite + Multi-Retrieval + RRF ---

[RETRIEVE] Original question: What is the capital of France?
  [REWRITE] Original : What is the capital of France?
  [REWRITE] Rewritten: What is the capital city of France?
  [BM25] Retrieved 9 candidates
  [DENSE] Retrieved 9 candidates


/tmp/ipykernel_2350/594244655.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'chunk_id': 7, 'source': 'manual', 'title': 'LangChain Overview'}, page_content='Vector stores are databases optimized for storing and searching embedding vectors.\nFAISS (Facebook AI Similarity Search) is a popular open-source vector store for fast nearest-neighbor search.\nOther vector stores include Chroma, Pinecone, Weaviate, and Qdrant.'), -0.3377804211859827), (Document(metadata={'source': 'manual', 'title': 'LangChain Overview', 'chunk_id': 5}, page_content='Embeddings are numerical vector representations of text. Similar texts produce similar embedding vectors.\nThis property allows us to find semantically similar documents using distance metrics like cosine similarity.'), -0.3738449736945726), (Document(metadata={'source': 'manual', 'chunk_id': 0, 'title': 'LangChain Overview'}, page_content='LangChain is a framework designed to simplify the creation of applicat

  [HyDE] Hypothetical passage: The capital of France is Paris. Known for its rich history, art, and culture, Paris is home to iconic landmarks such as the Eiffel Tower, the Louvre M...
  [HyDE] Retrieved 9 candidates
  [RRF] Fused into 9 unique candidates

--- Step 4: Cross-Encoder Reranking (top 5) ---


/tmp/ipykernel_2350/3098778183.py:27: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'source': 'manual', 'chunk_id': 7, 'title': 'LangChain Overview'}, page_content='Vector stores are databases optimized for storing and searching embedding vectors.\nFAISS (Facebook AI Similarity Search) is a popular open-source vector store for fast nearest-neighbor search.\nOther vector stores include Chroma, Pinecone, Weaviate, and Qdrant.'), -0.2770723534719819), (Document(metadata={'source': 'manual', 'title': 'LangChain Overview', 'chunk_id': 1}, page_content='LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.\nIt uses a graph-based architecture where nodes represent computation steps and edges represent transitions.'), -0.28704792245863153), (Document(metadata={'chunk_id': 2, 'source': 'manual', 'title': 'LangChain Overview'}, page_content='LangGraph is particularly useful for building complex agent

  Final context passages:
    Rank 1 | CE=-11.0118 | RAG stands for Retrieval-Augmented Generation. It is a technique where relevant ...
    Rank 2 | CE=-11.0686 | LangChain is a framework designed to simplify the creation of applications using...
    Rank 3 | CE=-11.0888 | LangGraph is a library built on top of LangChain that enables building stateful,...
    Rank 4 | CE=-11.1216 | Embeddings are numerical vector representations of text. Similar texts produce s...
    Rank 5 | CE=-11.1263 | LangSmith is a developer platform for debugging, testing, evaluating, and monito...

--- Step 5: Augmented Generation (gpt-4o-mini) ---

--- Step 6: Cache Store ---
  [CACHE] STORED       | TTL=300s | key: "What is the capital of France?"

[ANSWER]
I cannot answer based on the provided context.



---
## Section 11 — Ablation: Compare Naive vs Advanced RAG

Run the same question through a simple single-retriever pipeline and the
full advanced pipeline side-by-side to observe the quality difference.


In [23]:
# ---------------------------------------------------------------
# NAIVE BASELINE — plain dense retrieval, no reranking
# ---------------------------------------------------------------

naive_retriever = chroma_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

naive_prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the context below.
If you cannot answer from the context, say so.

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

from langchain_core.runnables import RunnablePassthrough

naive_chain = (
    {"context": naive_retriever | format_docs, "question": RunnablePassthrough()}
    | naive_prompt
    | generation_llm
    | StrOutputParser()
)

def naive_rag(question: str) -> str:
    print(f"[NAIVE RAG] Question: {question}")
    answer = naive_chain.invoke(question)
    print(f"[NAIVE RAG] Answer: {answer}")
    return answer


In [24]:
# ---------------------------------------------------------------
# SIDE-BY-SIDE COMPARISON
# ---------------------------------------------------------------

test_question = "How do agents decide which tools to use and in what order?"

print("\n" + "=" * 65)
print("NAIVE RAG (dense only, no reranking)")
print("=" * 65)
naive_answer = naive_rag(test_question)

print("\n" + "=" * 65)
print("ADVANCED RAG (BM25 + Dense + HyDE + RRF + Cross-Encoder)")
print("=" * 65)
advanced_answer = advanced_rag(test_question, verbose=False)

print("\n" + "=" * 65)
print("COMPARISON SUMMARY")
print("=" * 65)
print(f"Question : {test_question}")
print()
print("Naive answer:")
print(naive_answer)
print()
print("Advanced answer:")
print(advanced_answer)



NAIVE RAG (dense only, no reranking)
[NAIVE RAG] Question: How do agents decide which tools to use and in what order?
[NAIVE RAG] Answer: Agents decide which tools to use and in what order based on the user's query.

ADVANCED RAG (BM25 + Dense + HyDE + RRF + Cross-Encoder)

ADVANCED RAG PIPELINE
Question: How do agents decide which tools to use and in what order?

--- Step 0: Cache Lookup ---
  [CACHE] MISS         | cache_type=N/A | no match above threshold=0.92

--- Steps 1-3: Query Rewrite + Multi-Retrieval + RRF ---

[RETRIEVE] Original question: How do agents decide which tools to use and in what order?


/tmp/ipykernel_2350/594244655.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'chunk_id': 8, 'source': 'manual', 'title': 'LangChain Overview'}, page_content="Agents are LLM-powered systems that can use tools to accomplish tasks.\nA tool is a function that the agent can call, such as a web search, calculator, or API call.\nAgents decide which tools to use and in what order based on the user's query."), 0.4377219285892555), (Document(metadata={'source': 'manual', 'chunk_id': 2, 'title': 'LangChain Overview'}, page_content='LangGraph is particularly useful for building complex agent workflows that require cycles and branching logic.'), 0.12703171434416705), (Document(metadata={'source': 'manual', 'title': 'LangChain Overview', 'chunk_id': 0}, page_content='LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).\nIt provides tools for chaining prompts, managing memory, integrating with external d


--- Step 4: Cross-Encoder Reranking (top 5) ---

--- Step 5: Augmented Generation (gpt-4o-mini) ---

--- Step 6: Cache Store ---
  [CACHE] STORED       | TTL=300s | key: "How do agents decide which tools to use and in what order?"

[ANSWER]
Agents decide which tools to use and in what order based on the user's query.


COMPARISON SUMMARY
Question : How do agents decide which tools to use and in what order?

Naive answer:
Agents decide which tools to use and in what order based on the user's query.

Advanced answer:
Agents decide which tools to use and in what order based on the user's query.


---
## Section 12 — Interactive Query

Type any question and run the full advanced pipeline interactively.


In [ ]:
# ---------------------------------------------------------------
# INTERACTIVE QUERY
# ---------------------------------------------------------------

user_question = input("Enter your question: ").strip()

if user_question:
    final_answer = advanced_rag(
        user_question,
        top_k_retrieval=10,
        top_k_rerank=5,
        verbose=True
    )
else:
    print("[INFO] No question entered.")


---
## Final — Session Cache Statistics

Print a full summary of all cache lookups made during this session.


In [25]:
# ---------------------------------------------------------------
# FINAL CACHE STATISTICS
# ---------------------------------------------------------------
rag_cache.stats()



  CACHE STATISTICS
  Exact hits      : 2
  Semantic hits   : 0
  Misses          : 6
  Evictions (TTL) : 0
  Total lookups   : 8
  Hit rate        : 25.0%
  Live entries    : 6
  TTL setting     : 300s
  Sim threshold   : 0.92

